In [1]:
# --- IMPORTS ---
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import os
from datetime import datetime

In [2]:
# --- CONFIGURE CHROME OPTIONS ---
def get_chrome_options():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114 Safari/537.36"
    )
    return chrome_options

In [3]:
def create_driver():
    return webdriver.Chrome(options=get_chrome_options())

In [4]:
# --- SCRAPE DETAILS FROM PROPERTY PAGE ---
def prop_details_collect(link):
    driver = create_driver()
    driver.get(link)
    time.sleep(4)

    def safe_find(by, value, default=""):
        try:
            return driver.find_element(by, value).text.strip()
        except:
            return default

    def safe_find_list(by, value):
        try:
            return [e.text for e in driver.find_elements(by, value) if e.text and not e.text.startswith("No ")]
        except:
            return []

    details = {
        "Property ID": safe_find(By.ID, "Prop_Id"),
        "Address": safe_find(By.CLASS_NAME, "component__pdPropAddress").replace("in", "").strip(),
        "Society": safe_find(By.CSS_SELECTOR, 'div[data-label="ABOUT_SOCIETY"] span'),
        "Rating": safe_find(By.CLASS_NAME, "rc__ratingWrap"),
        "Furnishing": safe_find_list(By.CSS_SELECTOR, 'div[data-label="FURNISHING"] li'),
        "Facilities": safe_find_list(By.CSS_SELECTOR, 'div[data-label="FACILITIES"] li'),
        "Nearby": safe_find_list(By.CLASS_NAME, "NearByLocation__infoText"),
        "Overlooking" : safe_find_list(By.CLASS_NAME, "component__overlookingStyle")
    }

    try:
        table = driver.find_element(By.CLASS_NAME, "component__factTable")
        details.update({
            "Bedrooms": safe_find(By.ID, "bedRoomNum", "0"),
            "Bathrooms": safe_find(By.ID, "bathroomNum", "0"),
            "Balcony": safe_find(By.ID, "balconyNum", "0"),
            "Additional Rooms": safe_find(By.ID, "additionalRooms"),
            "Floor Number": safe_find(By.ID, "floorNumLabel", "0"),
            "Property Age/Possession": safe_find(By.ID, "agePossessionLbl"),
            "Facing": safe_find(By.ID, "facingLabel"),
        })
    except:
        details.update({
            "Bedrooms": "0",
            "Bathrooms": "0",
            "Balcony": "0",
            "Additional Rooms": "",
            "Floor Number": "0",
            "Property Age/Possession": "",
            "Facing": ""
        })
    print("Details collected for:", link)
    driver.quit()
    return details

In [5]:
# --- URL GENERATION ---
base_url = "https://www.99acres.com/resale-flats-in-kolkata-ffid"
urls = [f"{base_url}-page-{i}" for i in range(71, 101)]

In [6]:
# Define paths
base_dir = "output"
backup_dir = os.path.join(base_dir, "99acres_backups")
main_file = os.path.join(base_dir, "99acres_kolkata_scraped_data.csv")

# Create directories if they don't exist
os.makedirs(backup_dir, exist_ok=True)

# --- MAIN SCRAPER LOOP ---
data = []

for url in urls:
    driver = create_driver()
    driver.get(url)
    time.sleep(5)

    try:
        properties = driver.find_elements(By.CLASS_NAME, "tupleNew__platHTopaz")
        print(f"Found {len(properties)} properties on {url}")

        for prop in properties:
            try:
                name = prop.find_element(By.CLASS_NAME, "tupleNew__locationName").text
                link = prop.find_element(By.TAG_NAME, "a").get_attribute("href")
                flat_type = prop.find_element(By.CLASS_NAME, "tupleNew__bOld").text
                prices = prop.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text.replace("₹", "")
                persqft = prop.find_element(By.CSS_SELECTOR, ".tupleNew__perSqftWrap.ellipsis").text.replace("₹", "")
                area = prop.find_element(By.CLASS_NAME, "tupleNew__area1Type").text
                area_type = prop.find_element(By.CLASS_NAME, "tupleNew__areaType").text

                element = prop.find_element(By.CLASS_NAME, "tupleNew__propType")
                location = element.text.replace(flat_type, "").replace("in", "").strip()

                try:
                    highlights = prop.find_element(By.CLASS_NAME, "tupleNew__noRight").text.replace("Highlights :", "").strip()
                except:
                    highlights = ""

                try:
                    description = prop.find_element(By.CSS_SELECTOR, ".descPtag_undefined.tupleNew__descText").text
                except:
                    description = ""

                more = prop_details_collect(link)

                entry = {
                    "Project Name": name,
                    "Link": link,
                    "Flat Type": flat_type,
                    "Price": prices,
                    "Area": area,
                    "Area Type": area_type,
                    "Price per Sqft": persqft,
                    "Location": location,
                    "Highlights": highlights,
                    "Description": description,
                    **more
                }
                data.append(entry)
            except Exception as inner_e:
                print(f"⚠️ Property parse error: {inner_e}")

    except Exception as page_e:
        print(f"❌ Page error at {url}: {page_e}")
    finally:
        driver.quit()

    # ✅ Save partial data after each page
    print(f"Saving {len(data)} entries to file...")
    # Save main file
    df = pd.DataFrame(data)
    df.to_csv(main_file, index=False)
    
    # Save backup with timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    backup_path = os.path.join(backup_dir, f"99acres_data_{timestamp}.csv")
    df.to_csv(backup_path, index=False)
    

    print(f"🔁 Scraping: {url} ({urls.index(url)+1}/{len(urls)})")

print("🎉 All pages scraped and saved to '99acres_kolkata_scraped_data.csv'")

Found 0 properties on https://www.99acres.com/resale-flats-in-kolkata-ffid-page-71
Saving 0 entries to file...
🔁 Scraping: https://www.99acres.com/resale-flats-in-kolkata-ffid-page-71 (1/30)
Found 1 properties on https://www.99acres.com/resale-flats-in-kolkata-ffid-page-72
Details collected for: https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-action-area-1-kolkata-east-1650-sq-ft-spid-O82363522
Saving 1 entries to file...
🔁 Scraping: https://www.99acres.com/resale-flats-in-kolkata-ffid-page-72 (2/30)
Found 3 properties on https://www.99acres.com/resale-flats-in-kolkata-ffid-page-73
Details collected for: https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-primarc-southwinds-em-bypass-kolkata-south-1136-sq-ft-spid-J83196906
Details collected for: https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-new-barrakpur-kolkata-north-845-sq-ft-spid-K82621400
Details collected for: https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-primarc-